In [1]:
#@title Setup -- ablacoes (D vs D-1, janelas de lookback)
from pathlib import Path
PROJECT = Path(r'D:\tcc_ibov')
DIRS = {k: PROJECT / v for k, v in {
    'raw': 'data/raw', 'processed': 'data/processed',
    'models': 'models', 'reports': 'reports', 'utils': 'utils',
}.items()}

import sys; sys.path.insert(0, str(DIRS['utils']))
import importlib, tcc_utils; importlib.reload(tcc_utils)
from tcc_utils import make_target, walk_forward_splits

import random, os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (mean_absolute_error, f1_score,
                             matthews_corrcoef, brier_score_loss)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

class LSTMRegressor(nn.Module):
    """Identico ao nivel 4/5 (notebook 05): 1 camada, hidden=16, dropout=0.4."""
    def __init__(self, n_features, hidden=16, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden,
                            num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        last = self.dropout(out[:, -1, :])
        return self.head(last).squeeze(-1)

HIDDEN, DROPOUT, EPOCHS, LR, WD, BATCH = 16, 0.4, 75, 1e-3, 1e-4, 32
SEEDS = [0, 1, 2, 3, 4]

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))
def da(a, b):   return (np.sign(a) == np.sign(b)).mean()

def treina_avalia(X_tr, y_tr, X_te, seed):
    set_seed(seed)
    model = LSTMRegressor(X_tr.shape[-1], HIDDEN, DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    lossf = nn.MSELoss()
    Xtr = torch.tensor(X_tr, device=DEVICE); ytr = torch.tensor(y_tr, device=DEVICE)
    n = len(Xtr)
    model.train()
    for ep in range(EPOCHS):
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            b = perm[i:i+BATCH]
            opt.zero_grad()
            loss = lossf(model(Xtr[b]), ytr[b])
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_te, device=DEVICE)).cpu().numpy()
    return pred

def roda_lstm(X_day_full, y_raw_full, index_full, lookback, tag, k_pca=None, X_emb_full=None):
    """Pipeline generico: se k_pca/X_emb_full forem passados, aplica PCA por
    fold sobre X_emb_full e concatena com X_day_full (preco [+ sig]) antes de
    sequenciar. Caso contrario, X_day_full ja e o tensor diario completo.
    embargo = lookback (regra do projeto p/ qualquer janela testada)."""
    splits = walk_forward_splits(index_full, n_splits=5, min_train=252,
                                  expanding=True, embargo=lookback)
    registros, preds_media, var_exp = [], {}, []
    for kf, (tr_days, te_days) in enumerate(splits, 1):
        t0 = time.time()
        tr_set, te_set = set(tr_days), set(te_days)

        if k_pca is not None:
            tr_day_mask = np.array([i in tr_set for i in range(len(X_emb_full))])
            pca = PCA(n_components=k_pca, random_state=0).fit(X_emb_full[tr_day_mask])
            var_exp.append(pca.explained_variance_ratio_.sum())
            X_emb_pca = pca.transform(X_emb_full).astype(np.float32)
            X_day = np.concatenate([X_day_full, X_emb_pca], axis=1)
        else:
            X_day = X_day_full

        seqs, targets, sidx = [], [], []
        for i in range(lookback - 1, len(X_day)):
            seqs.append(X_day[i-lookback+1:i+1]); targets.append(y_raw_full[i]); sidx.append(i)
        seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

        mask_tr = np.array([i in tr_set for i in sidx])
        mask_te = np.array([i in te_set for i in sidx])
        X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
        X_te, y_te = seqs[mask_te], targets[mask_te]
        if len(X_tr) == 0 or len(X_te) == 0:
            print(f"  fold {kf}: sem dados suficientes (lookback={lookback}) -- pulando")
            continue

        n_f = X_tr.shape[-1]
        sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
        X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
        X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

        n_cal = max(int(len(X_tr) * 0.2), 1)
        preds_seeds = []
        for s in SEEDS:
            pred = treina_avalia(X_tr, y_tr, X_te, s)
            preds_seeds.append(pred)
            pred_tr = treina_avalia(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
            iso = IsotonicRegression(out_of_bounds='clip').fit(
                pred_tr, (y_tr[-n_cal:] > 0).astype(int))
            p_prob = iso.predict(pred)
            y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
            registros.append({
                'fold': kf, 'seed': s,
                'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
                'DA': da(y_te, pred),
                'F1': f1_score(y_bin, pred_dir, zero_division=0),
                'MCC': matthews_corrcoef(y_bin, pred_dir),
                'Brier': brier_score_loss(y_bin, p_prob),
            })
        preds_media[kf] = np.mean(preds_seeds, axis=0)
        print(f"  fold {kf}: {len(X_tr)} tr seqs | {time.time()-t0:.0f}s")

    df = pd.DataFrame(registros)
    np.savez(DIRS['reports'] / f'preds_{tag}.npz',
             **{f'lstm_f{k}': v for k, v in preds_media.items()})
    df.to_parquet(DIRS['reports'] / f'{tag}_full.parquet')
    return df

metric_cols = ['MAE', 'RMSE', 'DA', 'F1', 'MCC', 'Brier']
def resumo_desvio(df):
    r = df.groupby('fold')[metric_cols].mean().mean()
    d = ((df.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5
    return r, d

print("Setup OK.")


Device: cuda
Setup OK.


In [2]:
#@title Ablacao -- texto do dia D vs. D-1 (defasado), LSTM + BERTimbau k=8
# Testa se usar a noticia de ONTEM (D-1) para prever o retorno de D+1, em vez
# da noticia de HOJE (D), muda o resultado. Espera-se: ainda mais fraco que
# D, ja que o achado central (Secao 5.3 do texto) mostra que o texto de D
# correlaciona com o retorno de D (contemporaneo) -- textos mais antigos
# devem estar ainda mais dessincronizados do alvo D+1.
LOOKBACK_D1 = 21
K_PCA_D1 = 8

feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
feat_texto = pd.read_parquet(DIRS['processed'] / 'features_texto_bertimbau.parquet')
feat_texto = feat_texto.reindex(feat_preco.index)

EMB_COLS = [f'emb_{i:03d}' for i in range(768)]
SIG_COLS = ['n_noticias', 'emb_dispersao']
PRICE_COLS = [c for c in feat_preco.columns if c != 'y']

# Defasagem: texto[i] passa a ser o texto do dia i-1 (shift(1)); descarta o
# primeiro dia (sem D-1 valido) -- sem inventar dado, so reduz N em 1.
feat_texto_lag1 = feat_texto.shift(1)
valid = feat_texto_lag1['emb_000'].notna()
feat_preco_lag = feat_preco.loc[valid]
feat_texto_lag1 = feat_texto_lag1.loc[valid]
print(f"Dias apos remover o 1o (sem D-1): {len(feat_preco_lag)} "
      f"(esperado {len(feat_preco)-1})")

X_price_lag = feat_preco_lag[PRICE_COLS].values.astype(np.float32)
X_sig_lag   = feat_texto_lag1[SIG_COLS].values.astype(np.float32)
X_emb_lag   = feat_texto_lag1[EMB_COLS].values.astype(np.float32)
y_lag       = feat_preco_lag['y'].values.astype(np.float32)
X_day_lag   = np.concatenate([X_price_lag, X_sig_lag], axis=1)

print("=== LSTM + BERTimbau k=8, texto DEFASADO (D-1) ===")
reg_lag1 = roda_lstm(X_day_lag, y_lag, feat_preco_lag.index, LOOKBACK_D1,
                      'lstm_bertimbau_k8_lagD1', k_pca=K_PCA_D1, X_emb_full=X_emb_lag)

r_lag1, d_lag1 = resumo_desvio(reg_lag1)

# Correlacao direta: texto(D-1) -> retorno(D+1) [preditivo, defasado 2 dias
# do alvo] -- comparavel a texto(D) -> retorno(D+1) ja reportado no texto.
corr_lag1 = {}
for c in ['n_noticias', 'emb_dispersao']:
    corr_lag1[c] = np.corrcoef(feat_texto_lag1[c].values, y_lag)[0, 1]
print("\nCorrelacao texto(D-1) -> retorno(D+1):")
for c, v in corr_lag1.items():
    print(f"  {c}: {v:+.4f}")

n4 = pd.read_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
n5 = pd.read_parquet(DIRS['reports'] / 'lstm_texto_full.parquet')
res_n4, _ = resumo_desvio(n4)
res_n5, dev_n5 = resumo_desvio(n5)

print("\n=== MCC: preco | texto-D (nivel 5) | texto-D-1 (esta ablacao) ===")
print(f"  so-preco:      {res_n4['MCC']:+.4f}")
print(f"  texto D (k8):  {res_n5['MCC']:+.4f} +/- {dev_n5['MCC']:.4f}")
print(f"  texto D-1(k8): {r_lag1['MCC']:+.4f} +/- {d_lag1['MCC']:.4f}")
print("\nSalvos: preds_lstm_bertimbau_k8_lagD1.npz, lstm_bertimbau_k8_lagD1_full.parquet")


Dias apos remover o 1o (sem D-1): 1491 (esperado 1491)


=== LSTM + BERTimbau k=8, texto DEFASADO (D-1) ===


  fold 1: 232 tr seqs | 74s


  fold 2: 438 tr seqs | 31s


  fold 3: 644 tr seqs | 42s


  fold 4: 850 tr seqs | 53s


  fold 5: 1056 tr seqs | 67s



Correlacao texto(D-1) -> retorno(D+1):
  n_noticias: -0.0231
  emb_dispersao: -0.0356

=== MCC: preco | texto-D (nivel 5) | texto-D-1 (esta ablacao) ===
  so-preco:      -0.0086
  texto D (k8):  -0.0180 +/- 0.0538
  texto D-1(k8): -0.0158 +/- 0.0537

Salvos: preds_lstm_bertimbau_k8_lagD1.npz, lstm_bertimbau_k8_lagD1_full.parquet


In [3]:
#@title Ablacao -- janelas de lookback {1,3,5,10} -- LSTM so-preco
# O nivel 4 (notebook 05) fixou lookback=21 (alinhado as features de vol
# mais longas e a ACF). Aqui testamos se um lookback mais curto muda a
# conclusao de que o LSTM so-preco nao bate o acaso.
feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
PRICE_COLS = [c for c in feat_preco.columns if c != 'y']
X_price = feat_preco[PRICE_COLS].values.astype(np.float32)
y_preco = feat_preco['y'].values.astype(np.float32)

sweep_rows = []
for lb in [1, 3, 5, 10]:
    print(f"=== LSTM so-preco, lookback={lb} ===")
    df = roda_lstm(X_price, y_preco, feat_preco.index, lb, f'lstm_preco_lb{lb}')
    r, d = resumo_desvio(df)
    sweep_rows.append({'lookback': lb, 'fonte': 'so-preco', **r.to_dict(),
                        **{f'{m}_sd': d[m] for m in metric_cols}})

# referencia: lookback=21 ja calculado no notebook 05
n4 = pd.read_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
r21, d21 = resumo_desvio(n4)
sweep_rows.append({'lookback': 21, 'fonte': 'so-preco', **r21.to_dict(),
                    **{f'{m}_sd': d21[m] for m in metric_cols}})

sweep_preco_df = pd.DataFrame(sweep_rows).sort_values('lookback').set_index('lookback')
print("\n=== Sweep de lookback -- LSTM so-preco ===")
print(sweep_preco_df[['MAE','RMSE','DA','MCC','MCC_sd']].round(4).to_string())
sweep_preco_df.to_parquet(DIRS['reports'] / 'sweep_lookback_preco.parquet')
print("\nSalvo: sweep_lookback_preco.parquet")


=== LSTM so-preco, lookback=1 ===


  fold 1: 252 tr seqs | 15s


  fold 2: 458 tr seqs | 27s


  fold 3: 664 tr seqs | 38s


  fold 4: 870 tr seqs | 50s


  fold 5: 1076 tr seqs | 61s
=== LSTM so-preco, lookback=3 ===


... (12 outputs left out)

  fold 3: 655 tr seqs | 39s


  fold 4: 861 tr seqs | 51s


  fold 5: 1067 tr seqs | 62s



=== Sweep de lookback -- LSTM so-preco ===
             MAE    RMSE      DA     MCC  MCC_sd
lookback                                        
1         0.0122  0.0166  0.4992 -0.0138  0.0682
3         0.0123  0.0168  0.5019 -0.0058  0.0424
5         0.0125  0.0170  0.4957 -0.0034  0.0427
10        0.0125  0.0171  0.5056  0.0054  0.0450
21        0.0127  0.0172  0.4938 -0.0086  0.0567

Salvo: sweep_lookback_preco.parquet


In [4]:
#@title Ablacao -- janelas de lookback {1,3,5,10} -- LSTM + BERTimbau k=8
feat_texto = pd.read_parquet(DIRS['processed'] / 'features_texto_bertimbau.parquet')
feat_texto = feat_texto.reindex(feat_preco.index)
EMB_COLS = [f'emb_{i:03d}' for i in range(768)]
SIG_COLS = ['n_noticias', 'emb_dispersao']
X_sig = feat_texto[SIG_COLS].values.astype(np.float32)
X_emb = feat_texto[EMB_COLS].values.astype(np.float32)
X_day_txt = np.concatenate([X_price, X_sig], axis=1)

sweep_rows_txt = []
for lb in [1, 3, 5, 10]:
    print(f"=== LSTM + BERTimbau k=8, lookback={lb} ===")
    df = roda_lstm(X_day_txt, y_preco, feat_preco.index, lb, f'lstm_texto_k8_lb{lb}',
                    k_pca=8, X_emb_full=X_emb)
    r, d = resumo_desvio(df)
    sweep_rows_txt.append({'lookback': lb, 'fonte': 'BERTimbau-k8', **r.to_dict(),
                            **{f'{m}_sd': d[m] for m in metric_cols}})

n5 = pd.read_parquet(DIRS['reports'] / 'lstm_texto_full.parquet')
r21t, d21t = resumo_desvio(n5)
sweep_rows_txt.append({'lookback': 21, 'fonte': 'BERTimbau-k8', **r21t.to_dict(),
                        **{f'{m}_sd': d21t[m] for m in metric_cols}})

sweep_texto_df = pd.DataFrame(sweep_rows_txt).sort_values('lookback').set_index('lookback')
print("\n=== Sweep de lookback -- LSTM + BERTimbau k=8 ===")
print(sweep_texto_df[['MAE','RMSE','DA','MCC','MCC_sd']].round(4).to_string())
sweep_texto_df.to_parquet(DIRS['reports'] / 'sweep_lookback_texto.parquet')
print("\nSalvo: sweep_lookback_texto.parquet")


=== LSTM + BERTimbau k=8, lookback=1 ===


  fold 1: 252 tr seqs | 14s


  fold 2: 458 tr seqs | 25s


  fold 3: 664 tr seqs | 39s


  fold 4: 870 tr seqs | 54s


  fold 5: 1076 tr seqs | 62s
=== LSTM + BERTimbau k=8, lookback=3 ===


... (11 outputs left out)

  fold 2: 449 tr seqs | 30s


  fold 3: 655 tr seqs | 44s


  fold 4: 861 tr seqs | 53s


  fold 5: 1067 tr seqs | 69s

=== Sweep de lookback -- LSTM + BERTimbau k=8 ===
             MAE    RMSE      DA     MCC  MCC_sd
lookback                                        
1         0.0126  0.0170  0.4883  0.0007  0.0601
3         0.0128  0.0174  0.4950 -0.0206  0.0526
5         0.0128  0.0174  0.4957 -0.0116  0.0515
10        0.0130  0.0177  0.4928 -0.0056  0.0553
21        0.0132  0.0181  0.4944 -0.0180  0.0538

Salvo: sweep_lookback_texto.parquet


In [5]:
#@title Resumo consolidado das ablacoes (D vs D-1, sweep de lookback)
print("=== D vs. D-1 (BERTimbau k=8, lookback=21) -- MCC ===")
print(f"  so-preco:  {res_n4['MCC']:+.4f}")
print(f"  texto D:   {res_n5['MCC']:+.4f} +/- {dev_n5['MCC']:.4f}")
print(f"  texto D-1: {r_lag1['MCC']:+.4f} +/- {d_lag1['MCC']:.4f}")

print("\n=== Sweep de lookback -- MCC por janela ===")
comp = sweep_preco_df[['MCC','MCC_sd']].rename(columns={'MCC':'MCC_preco','MCC_sd':'MCC_preco_sd'}).join(
       sweep_texto_df[['MCC','MCC_sd']].rename(columns={'MCC':'MCC_texto','MCC_sd':'MCC_texto_sd'}))
print(comp.round(4).to_string())

comp.to_parquet(DIRS['reports'] / 'sweep_lookback_consolidado.parquet')
print("\nSalvo: sweep_lookback_consolidado.parquet")
print("\nConclusao preliminar (conferir apos rodar): se o MCC permanecer",
      "dentro do desvio entre sementes em TODAS as janelas, o resultado nulo",
      "e robusto a escolha do lookback -- nao e um artefato da janela de 21 dias.")


=== D vs. D-1 (BERTimbau k=8, lookback=21) -- MCC ===
  so-preco:  -0.0086
  texto D:   -0.0180 +/- 0.0538
  texto D-1: -0.0158 +/- 0.0537

=== Sweep de lookback -- MCC por janela ===
          MCC_preco  MCC_preco_sd  MCC_texto  MCC_texto_sd
lookback                                                  
1           -0.0138        0.0682     0.0007        0.0601
3           -0.0058        0.0424    -0.0206        0.0526
5           -0.0034        0.0427    -0.0116        0.0515
10           0.0054        0.0450    -0.0056        0.0553
21          -0.0086        0.0567    -0.0180        0.0538

Salvo: sweep_lookback_consolidado.parquet

Conclusao preliminar (conferir apos rodar): se o MCC permanecer dentro do desvio entre sementes em TODAS as janelas, o resultado nulo e robusto a escolha do lookback -- nao e um artefato da janela de 21 dias.


In [6]:
#@title Ablacao -- MLP simples sobre janela achatada (recorrencia vs. capacidade)
# Isola se a recorrencia da LSTM, per se, agrega desempenho frente a um MLP
# raso sobre a MESMA janela de 21 dias, apenas achatada (21*18=378 features)
# em vez de sequencial. Mesmo protocolo do nivel 4 (05_lstm): walk-forward,
# embargo=21, StandardScaler por fold, calibracao isotonica, 5 sementes,
# mesmo otimizador/epocas/dropout/wd -- unica diferenca e a arquitetura
# (MLP feed-forward vs. LSTM recorrente), para isolar o efeito da recorrencia.

class MLPRegressor(nn.Module):
    """MLP raso (1 camada oculta) sobre a janela achatada. hidden=32 (ordem
    de grandeza razoavel p/ 378 entradas -- casar o numero exato de
    parametros da LSTM nao faz sentido aqui, pois o formato de entrada e
    estruturalmente diferente: 18 features/passo recorrentes vs. 378 flat)."""
    def __init__(self, n_features_flat, hidden=32, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features_flat, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

HIDDEN_MLP = 32
LOOKBACK_MLP = 21

def treina_avalia_mlp(X_tr, y_tr, X_te, seed):
    """Identico a treina_avalia, mas achata a janela (N, lookback, feat) ->
    (N, lookback*feat) antes de alimentar o MLP."""
    set_seed(seed)
    X_tr_flat = X_tr.reshape(X_tr.shape[0], -1)
    X_te_flat = X_te.reshape(X_te.shape[0], -1)
    model = MLPRegressor(X_tr_flat.shape[-1], HIDDEN_MLP, DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    lossf = nn.MSELoss()
    Xtr = torch.tensor(X_tr_flat, device=DEVICE); ytr = torch.tensor(y_tr, device=DEVICE)
    n = len(Xtr)
    model.train()
    for ep in range(EPOCHS):
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            b = perm[i:i+BATCH]
            opt.zero_grad()
            loss = lossf(model(Xtr[b]), ytr[b])
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_te_flat, device=DEVICE)).cpu().numpy()
    return pred

feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
PRICE_COLS = [c for c in feat_preco.columns if c != 'y']
X_price = feat_preco[PRICE_COLS].values.astype(np.float32)
y_preco = feat_preco['y'].values.astype(np.float32)

splits_mlp = walk_forward_splits(feat_preco.index, n_splits=5, min_train=252,
                                  expanding=True, embargo=LOOKBACK_MLP)

registros_mlp, preds_media_mlp = [], {}
for kf, (tr_days, te_days) in enumerate(splits_mlp, 1):
    t0 = time.time()
    tr_set, te_set = set(tr_days), set(te_days)

    seqs, targets, sidx = [], [], []
    for i in range(LOOKBACK_MLP - 1, len(X_price)):
        seqs.append(X_price[i-LOOKBACK_MLP+1:i+1]); targets.append(y_preco[i]); sidx.append(i)
    seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

    mask_tr = np.array([i in tr_set for i in sidx])
    mask_te = np.array([i in te_set for i in sidx])
    X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
    X_te, y_te = seqs[mask_te], targets[mask_te]

    n_f = X_tr.shape[-1]
    sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
    X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
    X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

    n_cal = max(int(len(X_tr) * 0.2), 1)
    preds_seeds = []
    for s in SEEDS:
        pred = treina_avalia_mlp(X_tr, y_tr, X_te, s)
        preds_seeds.append(pred)
        pred_tr = treina_avalia_mlp(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
        iso = IsotonicRegression(out_of_bounds='clip').fit(
            pred_tr, (y_tr[-n_cal:] > 0).astype(int))
        p_prob = iso.predict(pred)
        y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
        registros_mlp.append({
            'fold': kf, 'seed': s,
            'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
            'DA': da(y_te, pred),
            'F1': f1_score(y_bin, pred_dir, zero_division=0),
            'MCC': matthews_corrcoef(y_bin, pred_dir),
            'Brier': brier_score_loss(y_bin, p_prob),
        })
    preds_media_mlp[kf] = np.mean(preds_seeds, axis=0)
    print(f"fold {kf}: {len(X_tr)} tr seqs | {time.time()-t0:.0f}s")

reg_mlp = pd.DataFrame(registros_mlp)
np.savez(DIRS['reports'] / 'preds_mlp_preco.npz',
         **{f'lstm_f{k}': v for k, v in preds_media_mlp.items()})
reg_mlp.to_parquet(DIRS['reports'] / 'mlp_preco_full.parquet')

# Comparacao MLP (janela achatada) vs. LSTM (mesma janela, recorrente) vs. XGBoost
n4 = pd.read_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
xgb = pd.read_parquet(DIRS['reports'] / 'baseline_xgboost_full.parquet')
r_mlp, d_mlp = resumo_desvio(reg_mlp)
r_lstm, d_lstm = resumo_desvio(n4)
r_xgb, d_xgb = resumo_desvio(xgb)

print("\n=== MLP (janela achatada) vs. LSTM (recorrente) vs. XGBoost -- MCC +/- dp entre sementes ===")
print(f"  MLP  : {r_mlp['MCC']:+.4f} +/- {d_mlp['MCC']:.4f}")
print(f"  LSTM : {r_lstm['MCC']:+.4f} +/- {d_lstm['MCC']:.4f}")
print(f"  XGB  : {r_xgb['MCC']:+.4f} +/- {d_xgb['MCC']:.4f}")
print("\n=== Demais metricas (media sobre folds) ===")
print(pd.DataFrame({'MLP': r_mlp, 'LSTM': r_lstm, 'XGBoost': r_xgb}).T[metric_cols].round(4).to_string())
print("\nSalvos: preds_mlp_preco.npz, mlp_preco_full.parquet")


fold 1: 232 tr seqs | 11s


fold 2: 438 tr seqs | 18s


fold 3: 644 tr seqs | 26s


fold 4: 850 tr seqs | 35s


fold 5: 1056 tr seqs | 41s

=== MLP (janela achatada) vs. LSTM (recorrente) vs. XGBoost -- MCC +/- dp entre sementes ===
  MLP  : -0.0042 +/- 0.0458
  LSTM : -0.0086 +/- 0.0567
  XGB  : +0.0401 +/- 0.0485

=== Demais metricas (media sobre folds) ===
            MAE    RMSE      DA      F1     MCC   Brier
MLP      0.0195  0.0270  0.4920  0.4817 -0.0042  0.2671
LSTM     0.0127  0.0172  0.4938  0.4147 -0.0086  0.2730
XGBoost  0.0117  0.0161  0.5153  0.4099  0.0401  0.2661

Salvos: preds_mlp_preco.npz, mlp_preco_full.parquet


In [7]:
#@title Ablacao -- anisotropia dos embeddings (whitening) + teste de permutacao
# Motivacao: embeddings de BERT sao conhecidamente anisotropicos (Ethayarajh,
# 2019) -- pares de textos SEM relacao semantica tendem a ter similaridade de
# cosseno bem acima de zero, porque boa parte da "energia" do espaco vive numa
# direcao/subespaco generico compartilhado, nao-informativo. O pipeline
# original (niveis 5/5b, Via 1/1b) aplica PCA direto sobre os embeddings
# CRUS, sem correcao dessa anisotropia -- risco real de que os primeiros
# componentes principais capturem majoritariamente esse artefato, nao sinal
# semantico, empurrando sinal genuino para componentes descartados no corte
# de k=8/k=30.
#
# Verificacao empirica (corpus completo, fora do pipeline por fold):
#   - similaridade de cosseno media entre pares aleatorios de manchetes: 0.66
#     (num espaco isotropico seria ~0). Confirma anisotropia real.
#   - o proprio teste de sanidade do notebook 03 ja mostrava isso sem que
#     tivesse sido notado: sim(bolsa em alta, receita de bolo) = 0.505 --
#     "deve ser BAIXA" no comentario original, mas 0.505 nao e baixo em termos
#     absolutos, so e menor que 0.899 (o par realmente relacionado).
#
# Correcao testada: whitening (decorrelacionar + igualar variancia em todas
# as direcoes, ao molde de BERT-flow) antes de agregar por dia e aplicar PCA.
# Isso e feito aqui de forma EXPLORATORIA (PCA/whitening ajustados uma vez
# sobre a serie completa, nao por fold) -- adequado para diagnostico de
# "existe sinal recuperavel aqui" mas NAO usado para treinar nenhum modelo
# reportado no trabalho (evitaria vazamento se fosse).

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from collections import defaultdict

corpus = pd.read_parquet(DIRS['processed'] / 'corpus_bucketed.parquet').reset_index(drop=True)
feat_preco_diag = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
y_diag = feat_preco_diag['y'].values
ret_hoje_diag = feat_preco_diag['ret_lag0'].values

def whiten(Emb):
    mu = Emb.mean(axis=0)
    cov = np.cov((Emb - mu).T)
    autoval, autovec = np.linalg.eigh(cov)
    autoval = np.clip(autoval, 1e-6, None)
    W = autovec @ np.diag(autoval**-0.5) @ autovec.T
    return (Emb - mu) @ W

def agrega_por_dia(Emb, dias_col):
    idx_por_dia = defaultdict(list)
    for i, d in enumerate(dias_col):
        idx_por_dia[d].append(i)
    dias_unicos = np.array(sorted(pd.unique(dias_col)))
    centroides = np.vstack([Emb[idx_por_dia[d]].mean(axis=0) for d in dias_unicos])
    return pd.DataFrame(centroides, index=pd.DatetimeIndex(dias_unicos))

def diagnostico(feat_dia, nome, k=100):
    X = feat_dia.reindex(feat_preco_diag.index).values.astype(np.float64)
    pca = PCA(n_components=k, random_state=0).fit(X)
    scores = pca.transform(X)
    corrs_pred = [abs(np.corrcoef(scores[:, i], y_diag)[0, 1]) for i in range(k)]
    corrs_cont = [abs(np.corrcoef(scores[:, i], ret_hoje_diag)[0, 1]) for i in range(k)]
    r_pred, r_cont = max(corrs_pred), max(corrs_cont)
    print(f"{nome} (var {pca.explained_variance_ratio_.sum():.1%}, k={k}): "
          f"max|r| preditivo={r_pred:.4f} | max|r| contemporaneo={r_cont:.4f}")
    return r_pred, scores

# --- Similaridade de cosseno media (checagem de anisotropia) ---
E_bert = np.load(DIRS['processed'] / 'emb_bertimbau_raw.npy')
rng = np.random.default_rng(0)
idx_amostra = rng.choice(len(E_bert), size=2000, replace=False)
amostra = E_bert[idx_amostra]
amostra_norm = amostra / np.linalg.norm(amostra, axis=1, keepdims=True)
sims = amostra_norm @ amostra_norm.T
iu = np.triu_indices(len(amostra), k=1)
print(f"Similaridade de cosseno media entre pares aleatorios de manchetes: "
      f"{sims[iu].mean():.3f} (0=isotropico; confirma anisotropia se >> 0)\n")

# --- BERTimbau: cru vs whitened, varios k ---
print("=== BERTimbau ===")
feat_cru = agrega_por_dia(E_bert, corpus['trading_day'].values)
feat_white_bert = agrega_por_dia(whiten(E_bert), corpus['trading_day'].values)
for k in (8, 30, 100, 300):
    diagnostico(feat_cru, f"  cru        k={k}", k=k)
    diagnostico(feat_white_bert, f"  whitened   k={k}", k=k)
print()

# --- FinBERT: cru vs whitened ---
print("=== FinBERT ===")
E_fb = np.load(DIRS['processed'] / 'finbert_emb_raw.npy')
feat_cru_fb = agrega_por_dia(E_fb, corpus['trading_day'].values)
feat_white_fb = agrega_por_dia(whiten(E_fb), corpus['trading_day'].values)
for k in (8, 30, 100, 300):
    diagnostico(feat_cru_fb, f"  cru        k={k}", k=k)
    diagnostico(feat_white_fb, f"  whitened   k={k}", k=k)

print("\nSanidade cruzada: a correlacao CONTEMPORANEA do FinBERT whitened "
      "(k>=30) deve aproximar-se de +0.201 -- a correlacao ja reportada via "
      "a cabeca de sentimento dedicada do FinBERT (Secao 5.4) -- confirmando "
      "que o whitening recupera sinal real, nao ruido.")


Similaridade de cosseno media entre pares aleatorios de manchetes: 0.659 (0=isotropico; confirma anisotropia se >> 0)

=== BERTimbau ===


  cru        k=8 (var 52.3%, k=8): max|r| preditivo=0.0409 | max|r| contemporaneo=0.0315
  whitened   k=8 (var 7.2%, k=8): max|r| preditivo=0.0346 | max|r| contemporaneo=0.0676


  cru        k=30 (var 72.1%, k=30): max|r| preditivo=0.0512 | max|r| contemporaneo=0.0637
  whitened   k=30 (var 16.4%, k=30): max|r| preditivo=0.0640 | max|r| contemporaneo=0.0672
  cru        k=100 (var 88.1%, k=100): max|r| preditivo=0.0871 | max|r| contemporaneo=0.0638


  whitened   k=100 (var 37.2%, k=100): max|r| preditivo=0.0745 | max|r| contemporaneo=0.0768


  cru        k=300 (var 97.8%, k=300): max|r| preditivo=0.0847 | max|r| contemporaneo=0.0881


  whitened   k=300 (var 73.6%, k=300): max|r| preditivo=0.0904 | max|r| contemporaneo=0.0680

=== FinBERT ===


  cru        k=8 (var 86.6%, k=8): max|r| preditivo=0.0761 | max|r| contemporaneo=0.2146
  whitened   k=8 (var 6.9%, k=8): max|r| preditivo=0.0262 | max|r| contemporaneo=0.1869
  cru        k=30 (var 92.4%, k=30): max|r| preditivo=0.0761 | max|r| contemporaneo=0.2146
  whitened   k=30 (var 16.1%, k=30): max|r| preditivo=0.0610 | max|r| contemporaneo=0.1888


  cru        k=100 (var 96.9%, k=100): max|r| preditivo=0.0826 | max|r| contemporaneo=0.2146
  whitened   k=100 (var 36.9%, k=100): max|r| preditivo=0.0600 | max|r| contemporaneo=0.1879


  cru        k=300 (var 99.4%, k=300): max|r| preditivo=0.0823 | max|r| contemporaneo=0.2146


  whitened   k=300 (var 73.3%, k=300): max|r| preditivo=0.0788 | max|r| contemporaneo=0.1886

Sanidade cruzada: a correlacao CONTEMPORANEA do FinBERT whitened (k>=30) deve aproximar-se de +0.201 -- a correlacao ja reportada via a cabeca de sentimento dedicada do FinBERT (Secao 5.4) -- confirmando que o whitening recupera sinal real, nao ruido.


In [8]:
#@title Teste de permutacao -- a correlacao preditiva whitened e real ou artefato de multiplas comparacoes?
# Com k grande (ex.: 300), o MAXIMO de varias correlacoes tende a crescer so
# por efeito de estatistica de ordem, mesmo sob H0 (nenhuma correlacao real).
# Testa isso diretamente: embaralha o alvo (quebra qualquer relacao real,
# preserva a estrutura interna dos componentes) e recalcula o max|r| --
# repete 1000x para construir a distribuicao nula empirica.

K_PERM = 300
pca_perm = PCA(n_components=K_PERM, random_state=0).fit(
    feat_white_bert.reindex(feat_preco_diag.index).values.astype(np.float64))
scores_perm = pca_perm.transform(
    feat_white_bert.reindex(feat_preco_diag.index).values.astype(np.float64))

obs_maxr = max(abs(np.corrcoef(scores_perm[:, i], y_diag)[0, 1]) for i in range(K_PERM))
print(f"max|r| observado (BERTimbau whitened, k={K_PERM}): {obs_maxr:.4f}")

rng = np.random.default_rng(0)
nulos = []
for _ in range(500):
    y_perm = rng.permutation(y_diag)
    nulos.append(max(abs(np.corrcoef(scores_perm[:, i], y_perm)[0, 1]) for i in range(K_PERM)))
nulos = np.array(nulos)
p_valor = (nulos >= obs_maxr).mean()
print(f"nulo empirico (500 permutacoes) -- media: {nulos.mean():.4f} | "
      f"p95: {np.percentile(nulos, 95):.4f}")
print(f"p-valor empirico: {p_valor:.3f}  "
      f"({'NAO significativo -- consistente com ruido' if p_valor > 0.05 else 'significativo'})")

resultado_anisotropia = pd.DataFrame([{
    'fonte': 'BERTimbau', 'k': K_PERM, 'max_r_observado': obs_maxr,
    'p_valor_permutacao': p_valor, 'nulo_media': nulos.mean(), 'nulo_p95': np.percentile(nulos, 95),
}])
resultado_anisotropia.to_parquet(DIRS['reports'] / 'ablacao_anisotropia_whitening.parquet')
print("\nSalvo: ablacao_anisotropia_whitening.parquet")


max|r| observado (BERTimbau whitened, k=300): 0.0904


nulo empirico (500 permutacoes) -- media: 0.0815 | p95: 0.1009
p-valor empirico: 0.178  (NAO significativo -- consistente com ruido)

Salvo: ablacao_anisotropia_whitening.parquet


In [9]:
#@title Ablacao -- qualidade do texto: manchetes-template e linguagem prospectiva/retrospectiva
# Duas hipoteses concretas sobre se a QUALIDADE do texto (nao a compressao)
# estaria suprimindo sinal:
# (1) Colunas fixas recorrentes (ex.: "Os 5 assuntos que vao movimentar o
#     mercado nesta terca-feira") tem embedding quase identico toda semana,
#     independente do conteudo real do dia -- vetores de informacao zero
#     diluindo a media diaria. Testa remover essas manchetes-template.
# (2) A maioria das manchetes e RETROSPECTIVA (recapitula o que ja aconteceu
#     -- reforca o sinal contemporaneo ja conhecido) e poucas sao
#     PROSPECTIVAS (linguagem de previsao/expectativa). Testa se isolar so
#     as prospectivas revela sinal preditivo escondido na maioria retrospectiva.

padroes_template = (r'os \d+ assuntos que v.o (movimentar|agitar) o(s)? mercado(s)? nesta|'
                     r'day trade hoje: o que esperar dos minicontratos')
mask_template = corpus['headline'].astype(str).str.contains(
    padroes_template, case=False, regex=True, na=False)
print(f"Manchetes-template identificadas: {mask_template.sum()} "
      f"({mask_template.mean():.1%} do corpus)\n")

prospectiva = (r'\b(deve|deverá|deveria|vai|irá|espera(-se)?|esperado|previsão|'
               r'previsões|projeta|projeção|estima|amanhã|próximo|próximos|'
               r'tende a|perspectiva|aponta para)\b')
retrospectiva = r'\b(fechou|encerrou|subiu|caiu|recuou|avançou|registrou|teve|ontem)\b'
mask_prosp = corpus['headline'].astype(str).str.contains(prospectiva, case=False, regex=True, na=False)
mask_retro = corpus['headline'].astype(str).str.contains(retrospectiva, case=False, regex=True, na=False)
print(f"Manchetes prospectivas: {mask_prosp.sum()} ({mask_prosp.mean():.1%})")
print(f"Manchetes retrospectivas: {mask_retro.sum()} ({mask_retro.mean():.1%})\n")

def agrega_filtrado_e_testa(mask_manter, nome, k=30, n_perm=300):
    sub_corpus = corpus[mask_manter].reset_index(drop=True)
    sub_E = E_bert[mask_manter.values]
    dias = sub_corpus['trading_day'].values
    idx_por_dia = defaultdict(list)
    for i, d in enumerate(dias):
        idx_por_dia[d].append(i)
    dias_validos = np.array(sorted(idx_por_dia.keys()))
    Ew = whiten(sub_E)
    centroides = np.vstack([Ew[idx_por_dia[d]].mean(axis=0) for d in dias_validos])
    feat = pd.DataFrame(centroides, index=pd.DatetimeIndex(dias_validos))
    feat_alvo = feat_preco_diag.reindex(feat.index)
    X = feat.values.astype(np.float64)
    y_sub = feat_alvo['y'].values
    ok = ~np.isnan(y_sub)
    X, y_sub = X[ok], y_sub[ok]
    kk = min(k, X.shape[0] - 1, X.shape[1])
    pca = PCA(n_components=kk, random_state=0).fit(X)
    scores = pca.transform(X)
    obs = max(abs(np.corrcoef(scores[:, i], y_sub)[0, 1]) for i in range(kk))

    rng_local = np.random.default_rng(0)
    nulos = np.array([
        max(abs(np.corrcoef(scores[:, i], rng_local.permutation(y_sub))[0, 1]) for i in range(kk))
        for _ in range(n_perm)
    ])
    p = (nulos >= obs).mean()
    print(f"{nome}: n_dias={len(y_sub)} | k={kk} | max|r| preditivo={obs:.4f} | "
          f"p-valor (permutacao, {n_perm}x)={p:.3f}")
    return obs, p

print("=== Filtragem de manchetes-template ===")
agrega_filtrado_e_testa(pd.Series(True, index=corpus.index), "COM templates (baseline)")
agrega_filtrado_e_testa(~mask_template, "SEM templates (filtrado)")

print("\n=== Prospectiva vs. retrospectiva ===")
agrega_filtrado_e_testa(mask_prosp, "SO prospectivas")
agrega_filtrado_e_testa(mask_retro, "SO retrospectivas")


Manchetes-template identificadas: 871 (1.5% do corpus)

Manchetes prospectivas: 5506 (9.6%)
Manchetes retrospectivas: 251 (0.4%)

=== Filtragem de manchetes-template ===


C:\Users\GAMER\AppData\Local\Temp\ipykernel_14068\624195851.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_template = corpus['headline'].astype(str).str.contains(
C:\Users\GAMER\AppData\Local\Temp\ipykernel_14068\624195851.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_prosp = corpus['headline'].astype(str).str.contains(prospectiva, case=False, regex=True, na=False)
C:\Users\GAMER\AppData\Local\Temp\ipykernel_14068\624195851.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_retro = corpus['headline'].astype(str).str.contains(retrospectiva, case=False, regex=True, na=False)


COM templates (baseline): n_dias=1492 | k=30 | max|r| preditivo=0.0640 | p-valor (permutacao, 300x)=0.347


SEM templates (filtrado): n_dias=1492 | k=30 | max|r| preditivo=0.0619 | p-valor (permutacao, 300x)=0.430

=== Prospectiva vs. retrospectiva ===


SO prospectivas: n_dias=1387 | k=30 | max|r| preditivo=0.0496 | p-valor (permutacao, 300x)=0.867


SO retrospectivas: n_dias=215 | k=30 | max|r| preditivo=0.1732 | p-valor (permutacao, 300x)=0.290


(np.float64(0.1731795845621845), np.float64(0.29))

In [10]:
#@title Interpretacao dos componentes de PCA (BERTimbau, k=8) -- diagnostico qualitativo
# Embeddings de BERT nao tem eixos individualmente interpretaveis como
# variaveis manuais (idade, renda etc.). A forma padrao de interpretar
# componentes de PCA sobre embeddings e (i) ver quais manchetes pontuam nos
# extremos de cada componente (leitura qualitativa) e (ii) correlacionar o
# componente, agregado por dia, com metadados conhecidos (leitura
# quantitativa). IMPORTANTE: PCA ajustado uma unica vez sobre a serie
# COMPLETA, so para fins de interpretacao -- nao e o PCA por fold usado nos
# modelos reportados (constituiria vazamento se fosse usado para treinar).

corpus_pca = pd.read_parquet(DIRS['processed'] / 'corpus_bucketed.parquet').reset_index(drop=True)
E_bert_pca = np.load(DIRS['processed'] / 'emb_bertimbau_raw.npy')

pca8 = PCA(n_components=8, random_state=0).fit(E_bert_pca)
scores8 = pca8.transform(E_bert_pca)
print(f"Variancia explicada por componente (k=8): "
      f"{[f'{v:.1%}' for v in pca8.explained_variance_ratio_]}")
print(f"Variancia acumulada: {pca8.explained_variance_ratio_.sum():.1%}\n")

# --- Leitura qualitativa: manchetes nos extremos de cada componente ---
N_EXTREMOS = 5
for comp in range(8):
    ordem = np.argsort(scores8[:, comp])
    print(f"--- Componente {comp} ---")
    print(f"  Score MAIS BAIXO ({N_EXTREMOS} manchetes):")
    for idx in ordem[:N_EXTREMOS]:
        print(f"    [{scores8[idx, comp]:+.2f}] {corpus_pca.loc[idx, 'headline'][:80]}")
    print(f"  Score MAIS ALTO ({N_EXTREMOS} manchetes):")
    for idx in ordem[-N_EXTREMOS:][::-1]:
        print(f"    [{scores8[idx, comp]:+.2f}] {corpus_pca.loc[idx, 'headline'][:80]}")
    print()

# --- Leitura quantitativa: correlacao do componente (agregado por dia) com
# metadados conhecidos ---
def agrega_scores_por_dia(scores, dias_col):
    idx_por_dia = defaultdict(list)
    for i, d in enumerate(dias_col):
        idx_por_dia[d].append(i)
    dias_unicos = np.array(sorted(pd.unique(dias_col)))
    centroides = np.vstack([scores[idx_por_dia[d]].mean(axis=0) for d in dias_unicos])
    return pd.DataFrame(centroides, index=pd.DatetimeIndex(dias_unicos),
                         columns=[f'comp_{i}' for i in range(scores.shape[1])])

scores_dia = agrega_scores_por_dia(scores8, corpus_pca['trading_day'].values)

feat_preco_meta = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
feat_sent_meta = pd.read_parquet(DIRS['processed'] / 'features_sentimento_finbert.parquet').reindex(feat_preco_meta.index)
feat_texto_meta = pd.read_parquet(DIRS['processed'] / 'features_texto_bertimbau.parquet').reindex(feat_preco_meta.index)
scores_dia = scores_dia.reindex(feat_preco_meta.index)

metadados = pd.DataFrame({
    'n_noticias': feat_texto_meta['n_noticias'],
    'emb_dispersao': feat_texto_meta['emb_dispersao'],
    'sent_polaridade': feat_sent_meta['sent_polaridade'],
    'ret_contemporaneo': feat_preco_meta['ret_lag0'],
    'ret_alvo_Dmais1': feat_preco_meta['y'],
    'dia_semana': feat_preco_meta.index.dayofweek,
}, index=feat_preco_meta.index)

print("=== Correlacao de cada componente (agregado por dia) com metadados ===")
tabela_corr = pd.DataFrame(index=[f'comp_{i}' for i in range(8)])
for col in metadados.columns:
    corrs = []
    for i in range(8):
        s = scores_dia[f'comp_{i}']
        m = metadados[col]
        ok = s.notna() & m.notna()
        corrs.append(np.corrcoef(s[ok], m[ok])[0, 1])
    tabela_corr[col] = corrs
print(tabela_corr.round(3).to_string())

tabela_corr.to_parquet(DIRS['reports'] / 'interpretacao_pca_componentes.parquet')
print("\nSalvo: interpretacao_pca_componentes.parquet")
print("\nNota: PCA ajustado sobre a serie completa, apenas para diagnostico -- "
      "os modelos reportados usam PCA ajustado por fold (Secao 4.2), como "
      "sempre neste trabalho.")


Variancia explicada por componente (k=8): ['11.8%', '5.9%', '5.2%', '4.0%', '2.9%', '2.6%', '2.4%', '2.1%']
Variancia acumulada: 36.9%

--- Componente 0 ---
  Score MAIS BAIXO (5 manchetes):
    [-3.77] Ibovespa descola do exterior e recua pressionado por Vale (VALE3) e bancos
    [-3.69] Fed: Mester vê &#8216;complacência&#8217; em relação ao avanço das expectativas 
    [-3.67] Fluxo cambial total no ano até 19 de agosto é positivo em US$ 20,891 bi, diz BC
    [-3.62] Hackers têm presa fácil com vários alertas ignorados pelo governo dos EUA
    [-3.62] Powell diz que Fed começará a reduzir balanço de ativos no próximo encontro
  Score MAIS ALTO (5 manchetes):
    [+2.80] O que a fraude contábil da Americanas representou para Magalu e Casas Bahia?
    [+2.80] Dirigente do Fed diz que não tem havido espiral de alta de preços neste momento 
    [+2.79] Mercados da China iniciam recuperação após temores de colapso
    [+2.74] Ibovespa futuro sobe em última sessão de semana marcada por in

=== Correlacao de cada componente (agregado por dia) com metadados ===
        n_noticias  emb_dispersao  sent_polaridade  ret_contemporaneo  ret_alvo_Dmais1  dia_semana
comp_0      -0.091          0.324           -0.022             -0.033           -0.001      -0.019
comp_1       0.433         -0.598            0.153              0.013           -0.011       0.045
comp_2       0.117         -0.033            0.042             -0.001            0.015       0.008
comp_3       0.268         -0.087            0.127             -0.001            0.030       0.017
comp_4      -0.157          0.057           -0.107             -0.023           -0.036      -0.003
comp_5       0.386          0.094            0.004             -0.025           -0.010       0.026
comp_6      -0.345          0.357           -0.121             -0.009           -0.006      -0.010
comp_7      -0.162         -0.079           -0.064             -0.024           -0.001      -0.009

Salvo: interpretacao_pca_componentes.

In [11]:
#@title Ablacao -- ultima manchete do dia (recencia) vs. media do dia (agregacao atual)
# O pipeline principal agrega o dia por MEDIA de todas as manchetes ate o
# corte de 17h. Dado o achado central do trabalho -- o sinal textual e
# reativo, correlacionando com o retorno do PROPRIO dia, nao do seguinte --
# e plausivel que a media dilua a manchete mais recente (mais proxima do
# fechamento, logo do que ha de mais "fresco" ainda nao amplamente
# precificado) com manchetes da manha, ja mais antigas relativamente ao
# fechamento. Testa-se aqui uma agregacao alternativa: usar SO a manchete
# mais recente do dia (maior ts_utc dentro do trading_day), em vez da
# media. Nao requer nova coleta -- mesma base de manchetes ja existente.

corpus_ult = pd.read_parquet(DIRS['processed'] / 'corpus_bucketed.parquet').reset_index(drop=True)
E_bert_ult = np.load(DIRS['processed'] / 'emb_bertimbau_raw.npy')
assert len(corpus_ult) == E_bert_ult.shape[0]

def agrega_ultima_manchete(E, corpus, ts_col='ts_utc'):
    dias = corpus['trading_day'].values
    ts = corpus[ts_col].values
    idx_por_dia = defaultdict(list)
    for i, d in enumerate(dias):
        idx_por_dia[d].append(i)
    dias_unicos = np.array(sorted(idx_por_dia.keys()))

    ultima_emb, n_not, index_dias = [], [], []
    for d in dias_unicos:
        idx = idx_por_dia[d]
        idx_max = max(idx, key=lambda i: ts[i])
        ultima_emb.append(E[idx_max])
        n_not.append(len(idx))
        index_dias.append(d)

    ultima_emb = np.vstack(ultima_emb)
    feat = pd.DataFrame(ultima_emb, index=pd.DatetimeIndex(index_dias),
                         columns=[f'emb_{i:03d}' for i in range(E.shape[1])])
    feat['n_noticias'] = n_not
    feat['emb_dispersao'] = 0.0  # nao aplicavel a um unico ponto -- mantido so p/ manter o formato
    feat.index.name = 'trading_day'
    return feat

feat_texto_ultima = agrega_ultima_manchete(E_bert_ult, corpus_ult)
feat_texto_ultima = feat_texto_ultima.reindex(feat_preco.index)
assert feat_texto_ultima['emb_000'].notna().all(), "ultima manchete com NaN apos alinhar"

EMB_COLS = [f'emb_{i:03d}' for i in range(768)]
SIG_COLS = ['n_noticias', 'emb_dispersao']
X_sig_ult = feat_texto_ultima[SIG_COLS].values.astype(np.float32)
X_emb_ult = feat_texto_ultima[EMB_COLS].values.astype(np.float32)
X_day_ult = np.concatenate([X_price, X_sig_ult], axis=1)

print("=== LSTM + BERTimbau k=8, ULTIMA manchete do dia (recencia) ===")
reg_ultima = roda_lstm(X_day_ult, y_preco, feat_preco.index, 21, 'lstm_bertimbau_k8_ultima',
                        k_pca=8, X_emb_full=X_emb_ult)
r_ult, d_ult = resumo_desvio(reg_ultima)

# Correlacao direta: ultima manchete do dia -> retorno(D+1), comparavel a
# media do dia -> retorno(D+1) ja reportada no texto (Secao 5.5).
X_emb_ult_pca8 = PCA(n_components=8, random_state=0).fit_transform(X_emb_ult)
corr_ult_pred = [abs(np.corrcoef(X_emb_ult_pca8[:, i], y_preco)[0, 1]) for i in range(8)]
print(f"\nCorrelacao ultima-manchete(PCA k=8) -> retorno(D+1): max|r|={max(corr_ult_pred):.4f}")

n5_ref = pd.read_parquet(DIRS['reports'] / 'lstm_texto_full.parquet')
res_n5_ref, dev_n5_ref = resumo_desvio(n5_ref)

print("\n=== MCC: preco | texto-MEDIA do dia (nivel 5) | texto-ULTIMA manchete (esta ablacao) ===")
print(f"  so-preco:          {res_n4['MCC']:+.4f}")
print(f"  texto media (k8):  {res_n5_ref['MCC']:+.4f} +/- {dev_n5_ref['MCC']:.4f}")
print(f"  texto ultima (k8): {r_ult['MCC']:+.4f} +/- {d_ult['MCC']:.4f}")
print("\nSalvos: preds_lstm_bertimbau_k8_ultima.npz, lstm_bertimbau_k8_ultima_full.parquet")


=== LSTM + BERTimbau k=8, ULTIMA manchete do dia (recencia) ===


  fold 1: 232 tr seqs | 16s


  fold 2: 438 tr seqs | 29s


  fold 3: 644 tr seqs | 45s


  fold 4: 850 tr seqs | 57s


  fold 5: 1056 tr seqs | 70s

Correlacao ultima-manchete(PCA k=8) -> retorno(D+1): max|r|=0.0571

=== MCC: preco | texto-MEDIA do dia (nivel 5) | texto-ULTIMA manchete (esta ablacao) ===
  so-preco:          -0.0086
  texto media (k8):  -0.0180 +/- 0.0538
  texto ultima (k8): -0.0228 +/- 0.0516

Salvos: preds_lstm_bertimbau_k8_ultima.npz, lstm_bertimbau_k8_ultima_full.parquet


In [12]:
#@title Ablacao -- Bag-of-Words (TF-IDF+SVD) vs. embedding denso (BERTimbau)
# Motivado por achado da literatura (FinMTEB -- Tang & Yang, EMNLP 2025): em
# tarefas de similaridade semantica financeira, um Bag-of-Words simples supera
# embeddings densos sofisticados. Testa-se aqui se o mesmo padrao se repete
# na tarefa deste trabalho (previsao de retorno), comparando uma
# representacao lexical esparsa (TF-IDF + SVD, k=8) contra a representacao
# densa ja usada (BERTimbau + PCA, k=8, nivel 5), sob protocolo IDENTICO --
# mesma arquitetura LSTM, mesmas sementes, mesmo k, mesmas manchetes; a unica
# diferenca isolada e o tipo de representacao textual. Simplificacao
# deliberada: usa-se so [preco | representacao textual k=8], sem n_noticias/
# dispersao (que nao tem analogo direto em TF-IDF), para manter a comparacao
# a mais pura possivel entre os dois tipos de representacao.
# ANTI-VAZAMENTO: tanto o vocabulario/IDF do TfidfVectorizer quanto a reducao
# de dimensionalidade (TruncatedSVD) sao ajustados exclusivamente nos dias de
# TREINO de cada fold -- mesma disciplina ja aplicada ao PCA sobre embeddings.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

_ACENTOS = {
    'a': 'à', 'e': 'é', 'a2': 'ã', 'ha': 'á',
    'ate': 'até', 'nao': 'não', 'ha2': 'há', 'ja': 'já',
    'so': 'só', 'tambem': 'também', 'voce': 'você',
    'voces': 'vocês', 'nos2': 'nós', 'sera': 'será',
    'serao': 'serão',
}
STOPWORDS_PT = set("""
a ao aos aquela aquelas aquele aqueles aquilo as com como da das de dela
delas dele deles depois do dos e ela elas ele eles em entre era eram essa
essas esse esses esta estamos estas estava estavam este esteja estejam esteve
estive estivemos estiveram estou eu foi fomos for foram fosse fossem fui
isso isto lhe lhes mais mas me mesmo meu meus minha minhas muito na nas
nem no nos nossa nossas nosso nossos num numa o os ou para pela pelas pelo
pelos por qual quando que quem se seja sejam sem seu seus somos
sua suas te tem tenho ter teu teus tinha tinham tive tivemos tivessem
todas todo todos tu tua tuas um uma vos
""".split())
STOPWORDS_PT |= {_ACENTOS['a'], _ACENTOS['e'], _ACENTOS['a2'], _ACENTOS['ha'],
                  _ACENTOS['ate'], _ACENTOS['nao'], _ACENTOS['ha2'], _ACENTOS['ja'],
                  _ACENTOS['so'], _ACENTOS['tambem'], _ACENTOS['voce'],
                  _ACENTOS['voces'], _ACENTOS['nos2'], _ACENTOS['sera'],
                  _ACENTOS['serao']}

corpus_bow = pd.read_parquet(DIRS['processed'] / 'corpus_bucketed.parquet')
texto_por_dia = corpus_bow.groupby('trading_day')['headline'].apply(lambda s: ' '.join(s.astype(str)))
docs = [texto_por_dia.get(d, '') for d in feat_preco.index]
n_vazios = sum(1 for d in docs if not d.strip())
print(f"Dias sem manchete no periodo de estudo: {n_vazios} de {len(docs)}")

K_BOW = 8
LOOKBACK_BOW = 21

splits_bow = walk_forward_splits(feat_preco.index, n_splits=5, min_train=252,
                                  expanding=True, embargo=LOOKBACK_BOW)

registros_bow, preds_media_bow, var_exp_bow = [], {}, []
for kf, (tr_days, te_days) in enumerate(splits_bow, 1):
    t0 = time.time()
    tr_set, te_set = set(tr_days), set(te_days)

    docs_tr = [docs[i] for i in range(len(docs)) if i in tr_set]
    vectorizer = TfidfVectorizer(max_features=5000, min_df=2, max_df=0.5,
                                  stop_words=list(STOPWORDS_PT), lowercase=True)
    Xtfidf_tr = vectorizer.fit_transform(docs_tr)
    Xtfidf_full = vectorizer.transform(docs)

    svd = TruncatedSVD(n_components=K_BOW, random_state=0)
    svd.fit(Xtfidf_tr)
    var_exp_bow.append(svd.explained_variance_ratio_.sum())
    X_bow_svd = svd.transform(Xtfidf_full).astype(np.float32)

    X_day = np.concatenate([X_price, X_bow_svd], axis=1)

    seqs, targets, sidx = [], [], []
    for i in range(LOOKBACK_BOW - 1, len(X_day)):
        seqs.append(X_day[i-LOOKBACK_BOW+1:i+1]); targets.append(y_preco[i]); sidx.append(i)
    seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

    mask_tr = np.array([i in tr_set for i in sidx])
    mask_te = np.array([i in te_set for i in sidx])
    X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
    X_te, y_te = seqs[mask_te], targets[mask_te]

    n_f = X_tr.shape[-1]
    sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
    X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
    X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

    n_cal = max(int(len(X_tr) * 0.2), 1)
    preds_seeds = []
    for s in SEEDS:
        pred = treina_avalia(X_tr, y_tr, X_te, s)
        preds_seeds.append(pred)
        pred_tr = treina_avalia(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
        iso = IsotonicRegression(out_of_bounds='clip').fit(
            pred_tr, (y_tr[-n_cal:] > 0).astype(int))
        p_prob = iso.predict(pred)
        y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
        registros_bow.append({
            'fold': kf, 'seed': s,
            'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
            'DA': da(y_te, pred),
            'F1': f1_score(y_bin, pred_dir, zero_division=0),
            'MCC': matthews_corrcoef(y_bin, pred_dir),
            'Brier': brier_score_loss(y_bin, p_prob),
        })
    preds_media_bow[kf] = np.mean(preds_seeds, axis=0)
    print(f"fold {kf}: {len(X_tr)} tr seqs | vocab={len(vectorizer.vocabulary_)} | SVD var {var_exp_bow[-1]:.1%} | {time.time()-t0:.0f}s")

reg_bow = pd.DataFrame(registros_bow)
np.savez(DIRS['reports'] / 'preds_lstm_bow_k8.npz',
         **{f'lstm_f{k}': v for k, v in preds_media_bow.items()})
reg_bow.to_parquet(DIRS['reports'] / 'lstm_bow_k8_full.parquet')

r_bow, d_bow = resumo_desvio(reg_bow)
print(f"\nVariancia explicada media (SVD sobre TF-IDF, k=8): {np.mean(var_exp_bow):.1%}")
print("\n=== MCC: preco | texto denso BERTimbau k8 (nivel 5) | texto BoW/TF-IDF k8 (esta ablacao) ===")
print(f"  so-preco:              {res_n4['MCC']:+.4f}")
print(f"  texto denso (k8):      {res_n5_ref['MCC']:+.4f} +/- {dev_n5_ref['MCC']:.4f}")
print(f"  texto BoW/TF-IDF (k8): {r_bow['MCC']:+.4f} +/- {d_bow['MCC']:.4f}")
print("\nSalvos: preds_lstm_bow_k8.npz, lstm_bow_k8_full.parquet")


Dias sem manchete no periodo de estudo: 0 de 1492


fold 1: 232 tr seqs | vocab=2903 | SVD var 7.2% | 18s


fold 2: 438 tr seqs | vocab=4739 | SVD var 6.0% | 30s


fold 3: 644 tr seqs | vocab=5000 | SVD var 6.0% | 45s


fold 4: 850 tr seqs | vocab=5000 | SVD var 6.7% | 63s


fold 5: 1056 tr seqs | vocab=5000 | SVD var 7.2% | 74s

Variancia explicada media (SVD sobre TF-IDF, k=8): 6.6%

=== MCC: preco | texto denso BERTimbau k8 (nivel 5) | texto BoW/TF-IDF k8 (esta ablacao) ===
  so-preco:              -0.0086
  texto denso (k8):      -0.0180 +/- 0.0538
  texto BoW/TF-IDF (k8): -0.0211 +/- 0.0626

Salvos: preds_lstm_bow_k8.npz, lstm_bow_k8_full.parquet


In [13]:
#@title Desambiguacao -- BoW/TF-IDF com k=30 (retem mais variancia)
# O SVD sobre TF-IDF com k=8 reteve so 6,6% da variancia (contra ~45-55% do
# BERTimbau no mesmo k=8) -- o espaco lexical (ate 5.000 termos) e muito mais
# disperso que o denso (768d), tornando k=8 uma compressao bem mais agressiva
# para o BoW. Mesmo controle ja aplicado ao BERTimbau (nivel 5b) e ao FinBERT
# (via 1b): se k=30 TAMBEM der nulo, reforca que a ausencia de sinal e real,
# nao um artefato de compressao excessiva. Reusa 'docs' ja computado na
# celula anterior; mesmo protocolo (walk-forward, embargo, LSTM, calibracao).

K_BOW30 = 30

splits_bow30 = walk_forward_splits(feat_preco.index, n_splits=5, min_train=252,
                                    expanding=True, embargo=LOOKBACK_BOW)

registros_bow30, preds_media_bow30, var_exp_bow30 = [], {}, []
for kf, (tr_days, te_days) in enumerate(splits_bow30, 1):
    t0 = time.time()
    tr_set, te_set = set(tr_days), set(te_days)

    docs_tr = [docs[i] for i in range(len(docs)) if i in tr_set]
    vectorizer = TfidfVectorizer(max_features=5000, min_df=2, max_df=0.5,
                                  stop_words=list(STOPWORDS_PT), lowercase=True)
    Xtfidf_tr = vectorizer.fit_transform(docs_tr)
    Xtfidf_full = vectorizer.transform(docs)

    svd = TruncatedSVD(n_components=K_BOW30, random_state=0)
    svd.fit(Xtfidf_tr)
    var_exp_bow30.append(svd.explained_variance_ratio_.sum())
    X_bow_svd30 = svd.transform(Xtfidf_full).astype(np.float32)

    X_day = np.concatenate([X_price, X_bow_svd30], axis=1)

    seqs, targets, sidx = [], [], []
    for i in range(LOOKBACK_BOW - 1, len(X_day)):
        seqs.append(X_day[i-LOOKBACK_BOW+1:i+1]); targets.append(y_preco[i]); sidx.append(i)
    seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

    mask_tr = np.array([i in tr_set for i in sidx])
    mask_te = np.array([i in te_set for i in sidx])
    X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
    X_te, y_te = seqs[mask_te], targets[mask_te]

    n_f = X_tr.shape[-1]
    sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
    X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
    X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

    n_cal = max(int(len(X_tr) * 0.2), 1)
    preds_seeds = []
    for s in SEEDS:
        pred = treina_avalia(X_tr, y_tr, X_te, s)
        preds_seeds.append(pred)
        pred_tr = treina_avalia(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
        iso = IsotonicRegression(out_of_bounds='clip').fit(
            pred_tr, (y_tr[-n_cal:] > 0).astype(int))
        p_prob = iso.predict(pred)
        y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
        registros_bow30.append({
            'fold': kf, 'seed': s,
            'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
            'DA': da(y_te, pred),
            'F1': f1_score(y_bin, pred_dir, zero_division=0),
            'MCC': matthews_corrcoef(y_bin, pred_dir),
            'Brier': brier_score_loss(y_bin, p_prob),
        })
    preds_media_bow30[kf] = np.mean(preds_seeds, axis=0)
    print(f"fold {kf}: {len(X_tr)} tr seqs | vocab={len(vectorizer.vocabulary_)} | SVD var {var_exp_bow30[-1]:.1%} | {time.time()-t0:.0f}s")

reg_bow30 = pd.DataFrame(registros_bow30)
np.savez(DIRS['reports'] / 'preds_lstm_bow_k30.npz',
         **{f'lstm_f{k}': v for k, v in preds_media_bow30.items()})
reg_bow30.to_parquet(DIRS['reports'] / 'lstm_bow_k30_full.parquet')

r_bow30, d_bow30 = resumo_desvio(reg_bow30)

# Comparacao final -- todas as representacoes de texto testadas ate aqui
n5_k30 = pd.read_parquet(DIRS['reports'] / 'lstm_texto_k30_full.parquet')
res_n5_k30, dev_n5_k30 = resumo_desvio(n5_k30)

print(f"\nVariancia explicada media (SVD sobre TF-IDF, k=30): {np.mean(var_exp_bow30):.1%} "
      f"(k=8 havia retido {np.mean(var_exp_bow):.1%})")
print("\n=== Comparacao final -- MCC +/- dp entre sementes ===")
linhas = [
    ("so-preco",                         res_n4['MCC'], None),
    ("BERTimbau denso k=8 (nivel 5)",    res_n5_ref['MCC'], dev_n5_ref['MCC']),
    ("BERTimbau denso k=30 (nivel 5b)",  res_n5_k30['MCC'], dev_n5_k30['MCC']),
    ("BoW/TF-IDF k=8 (esta secao)",      r_bow['MCC'], d_bow['MCC']),
    ("BoW/TF-IDF k=30 (esta secao)",     r_bow30['MCC'], d_bow30['MCC']),
]
for nome, mcc, dev in linhas:
    dev_str = f" +/- {dev:.4f}" if dev is not None else ""
    print(f"  {nome:<32s}: MCC {mcc:+.4f}{dev_str}")

print("\nSalvos: preds_lstm_bow_k30.npz, lstm_bow_k30_full.parquet")


fold 1: 232 tr seqs | vocab=2903 | SVD var 22.1% | 18s


fold 2: 438 tr seqs | vocab=4739 | SVD var 15.8% | 31s


fold 3: 644 tr seqs | vocab=5000 | SVD var 14.0% | 46s


fold 4: 850 tr seqs | vocab=5000 | SVD var 14.2% | 58s


fold 5: 1056 tr seqs | vocab=5000 | SVD var 14.5% | 71s

Variancia explicada media (SVD sobre TF-IDF, k=30): 16.1% (k=8 havia retido 6.6%)

=== Comparacao final -- MCC +/- dp entre sementes ===
  so-preco                        : MCC -0.0086
  BERTimbau denso k=8 (nivel 5)   : MCC -0.0180 +/- 0.0538
  BERTimbau denso k=30 (nivel 5b) : MCC -0.0085 +/- 0.0596
  BoW/TF-IDF k=8 (esta secao)     : MCC -0.0211 +/- 0.0626
  BoW/TF-IDF k=30 (esta secao)    : MCC -0.0157 +/- 0.0518

Salvos: preds_lstm_bow_k30.npz, lstm_bow_k30_full.parquet
